In [ ]:
import os
import subprocess as sp
from pathlib import Path

In [39]:
# Choose modules of interest to run

# Name of confluence offline module
# expanded and non_expanded modules each work from single 'setfinder' and 'combine_data' module
INCLUDED_MODULES = [
    # "expanded_setfinder",
    # "expanded_combine_data",
    # "input",
    # "non_expanded_setfinder",
    # "non_expanded_combine_data",
    # "prediagnostics",
    # # 'priors',
    
    "metroman",
    "metroman_consolidation",
    "unconstrained_momma",
    "constrained_momma",
    "hivdi",
    "sic4dvar",
    'busboi',
    "consensus",
    # # # # 'moi',
    # # # # 'unconstrained_offline',
    'validation',
    "output",
]

In [3]:
# Point to necessary directories
SIF_DIR = run_dir / "sif"  # Store built Docker images
sh_dir = run_dir / "sh_scripts"  # Store the sh scripts to run each module
report_dir = run_dir / "report"  # Job logs
mnt_dir = run_dir / f"{RUN_NAME}_mnt"  # t he mnt storing all confluence run data

os.environ["RUN_NAME"] = RUN_NAME
os.environ["BASE_DIR"] = str(BASE_DIR)
os.environ["SIF_DIR"] = str(SIF_DIR)

print(f"{RUN_NAME = }")
print(f"{SIF_DIR = }")

RUN_NAME = 'v4_short'
SIF_DIR = PosixPath('/nas/cee-water/cjgleason/ted/confluence/confluence_v4_short/sif')


## 3. Create Driver Script to run multiple modules

---
### Confluence Driver Script Generator (RUN ON HPC, NOT LOCALLY)
* Creates a batch submission script that will run all of your sif files in serial
* use sbatch to submit the entire run
* low resources and a long time should be used here, as all this job will do is launch your SLURM scripts you created for each module, it is basically a job manager

In [ ]:
import yaml
from jinja2 import Environment, FileSystemLoader

from pathlib import Path
import yaml
from jinja2 import Environment, FileSystemLoader

def load_config(config_path: str | Path) -> dict:
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def _validate_dir(dir: str | Path) -> Path:
    path = Path(dir)
    if not path.exists():
        raise FileNotFoundError(f"Directory not found: {path}")
    return path

def build_sifs_and_create_slurm_scripts(
    config_path: str | Path,
    template_dir: str | Path,
    run_list: list[str],
    included_modules: list[str],
    base_dir: str | Path,
    build: bool,
    continue_downloads: bool,
):
    config = load_config(config_path)
    base_dir = _validate_dir(base_dir)
    env = Environment(loader=FileSystemLoader(str(template_dir)), trim_blocks=True)

    module_template = env.get_template("sbatch.sh.j2")
    skip_flag = "-k" if continue_downloads else ""

    for run in run_list:
        run_dir = base_dir / f"confluence_{run}"
        mnt_dir = run_dir / f"{run}_mnt"
        
        dirs = {
            "sh": run_dir / "sh_scripts",
            "sif": run_dir / "sif",
            "report": run_dir / "report"
        }
        for d in dirs.values():
            d.mkdir(exist_ok=True, parents=True)

        built_images = set()

        for module_name in included_modules:
            if module_name not in config.get("modules", {}):
                raise KeyError(f"Module '{module_name}' is not defined in the configuration file.")
                
            module_cfg = config["modules"][module_name]
            time_limit = module_cfg.get("time", "00:20:00")
            mem_limit = module_cfg.get("mem", "4G")

            if build:
                image_name = (
                    module_name.replace("expanded_", "")
                    .replace("non_", "")
                    .replace("unconstrained_", "")
                    .replace("constrained_", "")
                )
                
                if image_name not in built_images:
                    sp.run(
                        [
                            "singularity",
                            "build",
                            "-F",
                            dirs["sif"] / f"{image_name}.simg",
                            f"docker://{config['docker']['username']}/{image_name}:{config['docker']['tag']}",
                        ],
                        check=True
                    )
                    built_images.add(image_name)

            # Explicit template path resolution from config
            command_template = env.get_template(module_cfg["template"])
            rendered_command = command_template.render(
                mnt_dir=mnt_dir,
                sif_dir=dirs["sif"],
                exp=config["experiment"],
                files=config["files"],
                module=module_cfg,
                skip_flag=skip_flag,
                run=run
            )

            rendered_script = module_template.render(
                job_name=f"{module_name}_{run}_cfl",
                report_dir=dirs["report"],
                module_name=module_name,
                time_limit=time_limit,
                mem_limit=mem_limit,
                hpc=config["hpc"],
                rendered_command=rendered_command
            )

            script_path = dirs["sh"] / f"{module_name}.sh"
            with open(script_path, "w") as file:
                file.write(rendered_script)

In [43]:
build_sifs_and_create_slurm_scripts(
    config_path = "/home/tlanghorst_umass_edu/cee-water/confluence/notebooks/config.yml",
    template_dir = "/home/tlanghorst_umass_edu/cee-water/confluence/notebooks/templates",
    run_list = [RUN_NAME],
    included_modules= INCLUDED_MODULES,
    base_dir = BASE_DIR,
    build = False,
    continue_downloads = True,
)

In [12]:
script_jobs

{'metroman_consolidation.sh': '6'}

In [8]:
scripts

['metroman.sh',
 'metroman_consolidation.sh',
 'unconstrained_momma.sh',
 'constrained_momma.sh',
 'hivdi.sh',
 'sic4dvar.sh',
 'busboi.sh',
 'consensus.sh']

In [44]:
# Create driver SLURM script for each run in run_list

# Define which modules have special (hardcoded) job counts
HARDCODED_JOBS = {
    "expanded_setfinder": "6",
    "expanded_combine_data": "1",
    "non_expanded_setfinder": "6",
    "non_expanded_combine_data": "1",
    "unconstrained_priors": "6",
    "constrained_priors": "6",
    "metroman_consolidation": "6",
    "output": "6",
}

job_name = str(run)
output_log_dir = f"{run_dir}/log"
partition = "ceewater_cjgleason-cpu"  # your partition here
time_limit = "7-00:00:00"
nodes = 1
ntasks = 1
cpus_per_task = 1
mem = "5G"

run = str(run)
directory = run_dir
sh_directory = f"{directory}/sh_scripts"
json_file = f"{directory}/{run}_mnt/input/reaches_of_interest.json"
expanded_json_file = (
    f"{directory}/{run}_mnt/input/expanded_reaches_of_interest.json"
)
reach_json_file = f"{directory}/{run}_mnt/input/reaches.json"
basin_json_file = f"{directory}/{run}_mnt/input/basin.json"
metroman_json_file = f"{directory}/{run}_mnt/input/metrosets.json"

batch_size = 1000  # cluster specific
concurrent_jobs = 400  # cluster specific

# Dynamically build script_jobs based on INCLUDED_MODULES
script_jobs = {}
for module in INCLUDED_MODULES:
    script_name = f"{module}.sh"

    if module in HARDCODED_JOBS:
        # Use hardcoded job count
        script_jobs[script_name] = HARDCODED_JOBS[module]

# Dynamically build scripts list (same order as INCLUDED_MODULES)
scripts = [f"{module}.sh" for module in INCLUDED_MODULES]

template_dir = Path("/nas/cee-water/cjgleason/ted/confluence/notebooks")
out_path = Path(f"{sh_directory}/slurm_driver.sh")

driver_script = generate_slurm_driver(
    template_dir = template_dir,
    output_filepath = out_path,
    job_name=job_name,
    output_log_dir=output_log_dir,
    partition=partition,
    time_limit=time_limit,
    nodes=nodes,
    ntasks=ntasks,
    cpus_per_task=cpus_per_task,
    mem=mem,
    run=run,
    directory=sh_directory,
    reach_chunks=20,
    json_file=json_file,
    expanded_json_file=expanded_json_file,
    reach_json_file=reach_json_file,
    basin_json_file=basin_json_file,
    metroman_json_file=metroman_json_file,
    batch_size=batch_size,
    concurrent_jobs=concurrent_jobs,
    script_jobs=script_jobs,
    scripts=scripts,
    # max_reaches = 2,
    dry_run = False
)

# # Optionally submit
sp.run(["sbatch", out_path], check=True)

Submitted batch job 59647914


CompletedProcess(args=['sbatch', PosixPath('/nas/cee-water/cjgleason/ted/confluence/confluence_v4_short/sh_scripts/slurm_driver.sh')], returncode=0)

---
# Reach or Module Changes

#### In order to run on different Type I reaches
* modify the file at /mnt/input/reaches_of_interest.json

#### In order to change a module and test it:
### Option 1
* change the module locally, build it and push to dockerhub using the first part of this notebook and then run as usual
* you can use the run_list variable to generate more submission script per moule to test more than one change at a time. However, whenver you submit them, they will still run one at a time, it just submits the next run automatically.
* Docker tag names highly recommended (custom_tag_name) for version control

### Option 2
* Use code below to generate everything in the HPC environment from cloning Git modules to running Confluence
* Docker images are built initially using GitHub container registry (ghcr.io/) and then overwritten with your HPC modules 
* This allows you to change module, re-build containers, and test as a module instantly
* Version control is handled by GitHub tag:
* Tag your local image
*      ! docker tag output:local ghcr.io/myGitAccount/moduleName:my-custom-tag
* Push to registry
*      docker push ghcr.io/myGitAccount/moduleName:my-custom-tag
* Modify tag name in function from 'latest' to 'my-custom-tag'

### Option 3
* Use a symlink to connect a previous run to a new directory
* Run module of interest using the data in previous modules (only need to run the changed module!)
* Combine 2 and 3 for efficient testing of multiple changes to one or many modules!

In [ ]:
#############################################

# Example Option 2:
## Run Confluence on an HPC end-to-end

## Requirements
* GitHub account
* apptainer installed on your HPC
* Basic python environment


## Overall Tasks
* Git clone all of the repos you want to run
* Prep an empty_mnt directory to store confluence run (requires gdown package in environment)
* Run image prep function to create the images from GitHub and your cloned modules
* Create SLURM submission scripts for each module
* Run the Confluence Driver Script Generator section of this notebook on your HPC to create a SLURM submission script that runs each of the modules one by one (the one click run)



---
## Functions (IGNORE)

In [12]:
from pathlib import Path
import json

input_dir = Path("/nas/cee-water/cjgleason/ted/confluence/confluence_v4_short/v4_short_mnt/input")
total_reaches = 0
for path in input_dir.glob("expanded_reaches_of_interest.json"):
    with open(path, 'r') as file:
        reaches = json.load(file)
    total_reaches += len(reaches)

total_reaches

14059

In [19]:
import netCDF4 as nc

ds = nc.Dataset('/nas/cee-water/cjgleason/ted/confluence/confluence_v4_short/v4_short_mnt/input/swot/12291300071_SWOT.nc')